In [1]:
using Pkg
Pkg.instantiate()
Pkg.update()

    Updating registry at `~/.julia/registries/General.toml`
    Updating git-repo `https://github.com/euriqa-brassboard/MSSim.jl.git`
     Project No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Project.toml`
    Manifest No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Manifest.toml`
        Info We haven't cleaned this depot up for a bit, running Pkg.gc()...
      Active manifest files: 9 found
      Active artifact files: 1 found
      Active scratchspaces: 0 found
     Deleted no artifacts, repos, packages or scratchspaces


In [2]:
include("sqrt_cz.jl")

opt_n! (generic function with 1 method)

In [3]:
using NPZ

In [4]:
Ω = 2π * 3
nseg = 30
nsubsample = 30
t_gate = 0.8
t_ramp = 0.02
opt = SplineOpt(Ω=Ω, nseg=nseg, nsubsample=nsubsample, t_gate=t_gate, t_ramp=t_ramp,
                lam_rob=0.1, lam_leak=0.7, lam_dark=0.7);
# opt = Opt(Ω=Ω, num_slices=num_slices, t_gate=t_gate,
#           lam_rob=0.1, lam_leak=1, lam_dark=1);
# optional keyword arguments:
# algorithm=:LD_CCSAQ, maxeval_pre=1000, maxtime=3, xtol=1e-7, minω=-2π * 10, maxω=2π * 10

In [5]:
best_obj, best_args = @time opt_n!(opt, 40; verbose=false, pre_threshold=0.002) # default verbosity is true

obj = 0.026118594976614444
obj = 0.009898734561291122
obj = 0.006236897732801662
obj = 0.001265695590376418
obj = 0.0003318644835724429
Round 20 done
obj = 0.00029840045440147135
Round 40 done
 14.123601 seconds (394.58 k allocations: 20.510 MiB, 1.72% compilation time)


(0.00029840045440147135, [34.031597303100504, -38.06854345636697, 12.4469645781338, 53.9887028296467, 29.67798842655606, 49.98599139596105, 2.9796344110187856, -44.38857516980265, -62.83185307179586, -62.83185307179586  …  41.77833615190637, -1.204096800696325, 27.551306466581146, 20.12182656435108, 56.0489106957091, 0.6169465543172662, -7.515393883732003, -9.987830002940173, 41.473074692881355, -4.975943642521685])

In [6]:
# More tries to refine the result.
for _ in 1:25
    best_obj, best_args = @time opt_n!(opt, 40; pre_threshold=0.002,
                                       verbose=false, best_obj=best_obj, best_args=best_args)
    if best_obj < 1e-6
        break
    end
end

obj = 2.327275098625322e-6
Round 20 done
Round 40 done
 19.872208 seconds (3.58 k allocations: 180.562 KiB, 0.04% compilation time)
Round 20 done
Round 40 done
 10.861306 seconds (752 allocations: 28.234 KiB)
Round 20 done
Round 40 done
 16.868412 seconds (778 allocations: 29.062 KiB)
Round 20 done
Round 40 done
 16.850510 seconds (776 allocations: 28.812 KiB)
obj = 4.824785808907007e-7
Round 20 done
Round 40 done
  9.197176 seconds (877 allocations: 36.125 KiB)


In [7]:
best_ϕs = fm_to_phase(opt, best_args)
println(best_ϕs)

[0.0, 0.04759493935930227, 0.09547300834636262, 0.1436302911870331, 0.19205895633301792, 0.24074725646187328, 0.2896795284770079, 0.3388361935076816, 0.38819375690900776, 0.43772480826195015, 0.48739802137332605, 0.5371781542758043, 0.5870260492279055, 0.6368986327140026, 0.6867489154443216, 0.7365259923549388, 0.7861750426077836, 0.8356373295906374, 0.8848502009171341, 0.933747088426759, 0.9822575081848494, 1.030307060482595, 1.0778174298370384, 1.1247063849910732, 1.170887778913446, 1.2162715487987537, 1.2607637160674467, 1.304266386365828, 1.3466777495660518, 1.3878920797661243, 1.4277997352899048, 1.4662871586871031, 1.5032395469378264, 1.538551532270751, 1.57212985236767, 1.6038933503634878, 1.6337729748462275, 1.6617117798570236, 1.6876649248901252, 1.7115996748928974, 1.7334954002658198, 1.753343576862484, 1.7711477859895994, 1.786923714406988, 1.800699154327587, 1.8125140034174483, 1.8224202647957366, 1.8304820470347336, 1.836775564159834, 1.8413891356495475, 1.8444231864354987

In [8]:
npzwrite("sqrtcz_0.6us_3MHz.npz", Dict("phase_list"=>best_ϕs, "t_gate"=>t_gate, "t_ramp"=>t_ramp, "Omega"=>Ω, "Omegas"=>Vector(opt.cb.Ωs)))